In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

# ============================
# USER SETTINGS (edit these)
# ============================

# CSV files with nucleotide indices for the two tips.
# Each file can contain indices separated by commas and/or newlines.
GROUP_A_CSV = "a_coord.txt"
GROUP_B_CSV = "b_coord.txt"

# Directory structure (relative to the umbrella root directory)
FORCES_DIR = "forces"
TRAJS_DIR  = "trajs"

# Trajectory file pattern: traj_win0.dat, traj_win1.dat, ...
TRAJ_PATTERN = "traj_win{win}.dat"

# Force file pattern: force_win0.txt, force_win1.txt, ...
FORCE_PATTERN = "force_win{win}.txt"

# Only read every Nth frame to speed up (1 = read all frames)
FRAME_STRIDE = 10

# Convert simulation length units to nm (oxDNA DNA model ~0.8518 nm/unit)
SIM_TO_NM = 0.8518

# ============================
# HELPER FUNCTIONS
# ============================

def load_indices_from_csv(path):
    """
    Load nucleotide indices from a CSV or text file.
    Accepts any combination of commas, spaces, and newlines.
    Returns a 1D numpy array of ints.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Index file not found: {path}")
    with open(path) as f:
        text = f.read()
    # Replace common separators with spaces, then split
    for sep in [",", ";", "\t"]:
        text = text.replace(sep, " ")
    tokens = [t for t in text.split() if t.strip()]
    indices = np.array([int(t) for t in tokens], dtype=int)
    return indices


def parse_oxdna_traj(filepath, frame_stride=1):
    """
    Generator that yields position arrays (shape [N, 3]) for each frame
    from an oxDNA trajectory/conf file.
    Assumes frames start with a line beginning 't =' followed by a box line.
    """
    with open(filepath) as f:
        lines = f.readlines()

    i = 0
    frame_idx = 0
    while i < len(lines):
        if lines[i].startswith("t ="):
            # line i: 't = ...'
            # line i+1: box line
            i += 2
            positions = []
            while i < len(lines) and not lines[i].startswith("t ="):
                vals = lines[i].split()
                if len(vals) >= 3:
                    positions.append([float(v) for v in vals[:3]])
                i += 1
            # yield only every frame_stride-th frame
            if frame_idx % frame_stride == 0 and positions:
                yield np.array(positions)
            frame_idx += 1
        else:
            i += 1


def compute_com_distance(positions, idx_A, idx_B):
    """
    Compute COM–COM distance for two groups of indices in a positions array.
    positions: [N, 3]
    idx_A, idx_B: 1D arrays of indices
    """
    com_A = positions[idx_A].mean(axis=0)
    com_B = positions[idx_B].mean(axis=0)
    return np.linalg.norm(com_A - com_B)


def get_r0_from_force_file(force_file):
    """
    Parse a force file (forces/force_winX.txt) and extract the line 'r0 = ...'.
    Returns float.
    """
    if not os.path.isfile(force_file):
        raise FileNotFoundError(f"Force file not found: {force_file}")
    with open(force_file) as f:
        for line in f:
            line_stripped = line.strip()
            if line_stripped.startswith("r0"):
                # expected format: r0 = value
                parts = line_stripped.replace(" ", "").split("=")
                if len(parts) == 2:
                    try:
                        return float(parts[1])
                    except ValueError:
                        pass
    raise ValueError(f"Could not find a valid 'r0' line in {force_file}")


# ============================
# MAIN LOGIC
# ============================

# Load tip indices
idx_A = load_indices_from_csv(GROUP_A_CSV)
idx_B = load_indices_from_csv(GROUP_B_CSV)

print(f"Loaded {len(idx_A)} indices for GROUP A from {GROUP_A_CSV}")
print(f"Loaded {len(idx_B)} indices for GROUP B from {GROUP_B_CSV}")

# Detect how many windows we have by listing force files
force_files = sorted(glob.glob(os.path.join(FORCES_DIR, "force_win*.txt")))

if not force_files:
    raise RuntimeError(f"No force files found in {FORCES_DIR}/force_win*.txt")

# Extract window indices from filenames: force_winX.txt
def window_index_from_filename(fname):
    base = os.path.basename(fname)
    # e.g. "force_win12.txt" -> "12"
    inside = base.replace("force_win", "").replace(".txt", "")
    return int(inside)

windows = sorted(window_index_from_filename(f) for f in force_files)
print(f"Detected windows: {windows}")

all_distances = []  # list of 1D arrays, one per window
r0_values = []      # list of target distances per window

for win in windows:
    # Force file and r0
    force_file = os.path.join(FORCES_DIR, FORCE_PATTERN.format(win=win))
    r0 = get_r0_from_force_file(force_file)
    r0_values.append(r0)

    # Trajectory file
    traj_file = os.path.join(TRAJS_DIR, TRAJ_PATTERN.format(win=win))
    if not os.path.isfile(traj_file):
        raise FileNotFoundError(f"Trajectory file not found for window {win}: {traj_file}")

    print(f"Processing window {win}: r0 = {r0:.3f} (sim units), traj = {traj_file}")

    distances = []

    for frame_positions in parse_oxdna_traj(traj_file, frame_stride=FRAME_STRIDE):
        # Sanity check: indices must be within range
        n_particles = frame_positions.shape[0]
        if idx_A.max() >= n_particles or idx_B.max() >= n_particles:
            raise IndexError(
                f"Indices in group A/B exceed number of particles ({n_particles}) "
                f"in window {win} trajectory."
            )
        d = compute_com_distance(frame_positions, idx_A, idx_B)
        distances.append(d)

    distances = np.array(distances)
    all_distances.append(distances)
    print(f"  Collected {len(distances)} distances for window {win}")

r0_values = np.array(r0_values)

# ============================
# PLOT BOX PLOTS
# ============================

# Sort windows by r0 just in case
order = np.argsort(r0_values)
r0_sorted = r0_values[order]
data_sorted = [all_distances[i] for i in order]

# Convert x positions to nm (optional)
x_positions_nm = r0_sorted * SIM_TO_NM

# Estimate a reasonable box width from spacing
if len(r0_sorted) > 1:
    dr_nm = np.min(np.diff(np.sort(x_positions_nm)))
    width_nm = dr_nm * 0.6
else:
    width_nm = 0.1  # arbitrary for single window

plt.figure(figsize=(10, 5))

plt.boxplot(
    data_sorted,
    positions=x_positions_nm,
    widths=width_nm,
    showfliers=False
)

plt.xlabel("Tip-to-tip distance (nm)")
plt.ylabel("Sampled distances (sim units)")
plt.title("Umbrella Windows: Distance Distributions per Window\n(via COM–COM distance)")
plt.grid(True, axis='y', alpha=0.3)

# Mark the nominal r0 positions (just as ticks)
plt.xticks(x_positions_nm, [f"{x:.1f}" for x in x_positions_nm], rotation=45)

plt.tight_layout()
plt.show()